# Что такое машинное обучение?

**Автор:** ИИ-агент, преподаватель по машинному обучению  
**Уровень:** начинающий (достаточно базового знания Python)  

После прохождения этого ноутбука вы:
- поймёте, чем машинное обучение отличается от обычного программирования;
- увидите, почему некоторые задачи невозможно решить «в лоб» кодом;
- разберётесь в ключевых понятиях: модель, обучение, входы, веса, прогнозы;
-亲手 построите мини-классификатор и проследите весь путь от данных до результата;
- научитесь распознавать, когда задача требует ML, а когда — обычного кода.

---

## План урока

1. **Подготовка окружения** — импорт библиотек, проверка версий.
2. **Обычное программирование vs машинное обучение** — наглядное сравнение.
3. **Ключевые концепции ML** — модель, веса, функция потерь, обучение.
4. **Базовый пример: классификатор «с нуля»** — строим простую модель вручную.
5. **Углубление: как обучение связано с историей ML** — от перцептрона до deep learning.
6. **От ML к глубокому обучению** — зачем нужны нейросети и слои.
7. **Практические задания / эксперименты** — 5 задач для самостоятельной работы.
8. **Идеи для дальнейшего изучения** — куда двигаться дальше.

---
## Шаг 1. Подготовка окружения

Для этого ноутбука нам понадобятся только базовые библиотеки: NumPy для вычислений, Matplotlib для визуализации и scikit-learn для генерации данных. Никакого PyTorch или TensorFlow — мы начнём с самых основ, чтобы понять суть.

In [ ]:
import sys
print(f'Python: {sys.version}')

import numpy as np
print(f'NumPy: {np.__version__}')

import matplotlib.pyplot as plt
print(f'Matplotlib готов')

import sklearn
print(f'scikit-learn: {sklearn.__version__}')

# Настройка визуализации
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12
print('\nОкружение готово!')

---
## Шаг 2. Обычное программирование vs машинное обучение

Давайте начнём с ключевого вопроса: **чем машинное обучение отличается от обычного программирования?**

В обычном программировании мы пишем **правила** (алгоритм), которые преобразуют **входы** в **результаты**. Мы думаем: «Если бы я делал это вручную, какие шаги бы выполнил?» — и переводим их в код.

Например, сортировка списка — это классическая задача обычного программирования. Мы точно знаем шаги:

In [ ]:
# Обычное программирование: мы пишем ПРАВИЛА

def sort_list(inputs):
    """Сортировка пузырьком — мы точно знаем шаги."""
    arr = inputs.copy()
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

# Входы -> Правила -> Результаты
unsorted = [64, 34, 25, 12, 22, 11, 90]
sorted_result = sort_list(unsorted)
print(f'Входы:  {unsorted}')
print(f'Результат: {sorted_result}')

Это работает прекрасно, потому что мы можем описать алгоритм сортировки точно и полностью. Но теперь представьте: **как бы вы написали программу, которая отличает кошку от собаки на фотографии?**

Вы могли бы начать:
- «Если у животного острые уши — это кошка» — но у некоторых собак тоже острые уши.
- «Если животное маленькое — это кошка» — но есть маленькие собаки и большие кошки.
- «Если у животного длинные усы — это кошка» — но усы бывают и у собак.

Каждое правило имеет исключения, и их комбинация превращается в бесконечный каскад `if-else`, который никогда не будет работать достаточно хорошо. Проблема в том, что визуальный мир слишком сложен, чтобы описать его конечным набором правил, написанных вручную.

**Вот тут и появляется машинное обучение.**

In [ ]:
# Наглядная демонстрация: попробуйте написать «правила» для классификации

# Представим, что у нас есть «фотографии» в виде числовых признаков:
# признак 1: размер ушей (0-1)
# признак 2: длина шерсти (0-1)
# признак 3: вес (0-1)

np.random.seed(42)

# Генерируем «кошек» и «собак»
n_animals = 50

# Кошки: обычно меньше, короче шерсть, разнообразные уши
cats = np.column_stack([
    np.random.uniform(0.1, 0.6, n_animals),   # размер ушей
    np.random.uniform(0.0, 0.5, n_animals),   # длина шерсти
    np.random.uniform(0.0, 0.4, n_animals),   # вес
])

# Собаки: больше, длиннее шерсть, уши разнообразнее
dogs = np.column_stack([
    np.random.uniform(0.3, 0.9, n_animals),   # размер ушей
    np.random.uniform(0.2, 1.0, n_animals),   # длина шерсти
    np.random.uniform(0.4, 1.0, n_animals),   # вес
])

# Визуализируем
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
features = ['Размер ушей', 'Длина шерсти', 'Вес']

for i, (ax, fname) in enumerate(zip(axes, features)):
    ax.scatter(cats[:, i], np.zeros(n_animals), alpha=0.6, label='Кошки', s=40)
    ax.scatter(dogs[:, i], np.ones(n_animals), alpha=0.6, label='Собаки', s=40)
    ax.set_xlabel(fname)
    ax.set_ylabel('Класс (0=кошка, 1=собака)')
    ax.legend()
    ax.set_title(f'{fname}: можно разделить?')

plt.tight_layout()
plt.show()

print('Видно: по каждому отдельному признаку классы ПЕРЕКРЫВАЮТСЯ.')
print('Нельзя написать простое правило вида «если вес > 0.5, то собака».')

Итак, в обычном программировании парадигма такая:

```
Входы  +  Правила (написанные вами)  ->  Результаты
```

А в машинном обучении — наоборот:

```
Входы  +  Результаты (из примеров)  ->  Правила (найденные машиной)
```

Мы даём компьютеру примеры (входы + правильные ответы), и он **сам** находит правила. Эти «правила» — это и есть **обученная модель**. Именно так работает классификатор кошек и собак: вы не пишете правила вручную — вы даёте тысячи фотографий с метками, и модель сама обнаруживает закономерности.

Это фундаментальная идея, которую сформулировал Артур Сэмюэл в 1959 году: *«Machine learning — это область исследований, которая даёт компьютерам способность учиться без явного программирования».*

In [ ]:
# Визуализация двух парадигм

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Обычное программирование
axes[0].annotate('Входы', xy=(0.15, 0.5), fontsize=16, ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightblue', ec='blue', lw=2))
axes[0].annotate('Правила\n(написаны вами)', xy=(0.5, 0.5), fontsize=14, ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='orange', lw=2))
axes[0].annotate('Результаты', xy=(0.85, 0.5), fontsize=16, ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightgreen', ec='green', lw=2))
axes[0].annotate('', xy=(0.35, 0.5), xytext=(0.25, 0.5),
                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
axes[0].annotate('', xy=(0.72, 0.5), xytext=(0.62, 0.5),
                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
axes[0].set_title('Обычное программирование', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Машинное обучение
axes[1].annotate('Входы', xy=(0.1, 0.7), fontsize=14, ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightblue', ec='blue', lw=2))
axes[1].annotate('Результаты\n(примеры)', xy=(0.1, 0.3), fontsize=14, ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightgreen', ec='green', lw=2))
axes[1].annotate('Правила\n(модель\nнаходит сама)', xy=(0.55, 0.5), fontsize=14, ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='orange', lw=2))
axes[1].annotate('', xy=(0.4, 0.55), xytext=(0.2, 0.65),
                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
axes[1].annotate('', xy=(0.4, 0.45), xytext=(0.2, 0.35),
                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
axes[1].set_title('Машинное обучение', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## Шаг 3. Ключевые концепции ML

Давайте разберём ключевые понятия машинного обучения на конкретном примере. Представьте, что мы хотим предсказать цену квартиры по её площади. Это классическая задача **регрессии**.

### Основные термины

| Термин | Что это | Пример |
|--------|----------|--------|
| **Входы (inputs)** | Данные, на основе которых делаем прогноз | Площадь квартиры (м2) |
| **Целевая переменная (target)** | То, что хотим предсказать | Цена квартиры (руб.) |
| **Модель** | Математическая функция, которая преобразует входы в прогноз | цена = w * площадь + b |
| **Веса (weights)** | Числа внутри модели, которые настраиваются в ходе обучения | w, b |
| **Функция потерь (loss)** | Измеряет, насколько прогнозы модели отличаются от реальности | Среднеквадратичная ошибка |
| **Обучение** | Процесс подбора весов, при котором loss минимизируется | Градиентный спуск |
| **Прогноз (prediction)** | Результат работы модели на новых данных | Предсказанная цена |

Модель — это не магия. Это просто функция с настраиваемыми параметрами. Обучение — это поиск таких параметров, при которых прогнозы модели лучше всего совпадают с правильными ответами из обучающих данных.

In [ ]:
# Демонстрация: линейная модель для предсказания цены квартиры

np.random.seed(42)

# Генерируем «данные о квартирах»
area = np.random.uniform(20, 120, 100)  # площадь от 20 до 120 м2
# «Истинная» зависимость: цена = 150_000 * площадь + 500_000 + шум
true_w = 150_000
true_b = 500_000
price = true_w * area + true_b + np.random.normal(0, 500_000, 100)

print(f'Сгенерировано {len(area)} квартир')
print(f'Площадь: от {area.min():.0f} до {area.max():.0f} м2')
print(f'Цена: от {price.min()/1e6:.1f} до {price.max()/1e6:.1f} млн руб.')

# Визуализация
plt.scatter(area, price / 1e6, alpha=0.5, s=30)
plt.xlabel('Площадь (м2)')
plt.ylabel('Цена (млн руб.)')
plt.title('Данные: цена квартиры vs площадь')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Обучаем простейшую линейную модель: цена = w * площадь + b
# Используем градиентный спуск вручную!

# Инициализируем веса случайным образом
w = np.random.randn() * 1000  # начальное значение веса
b = np.random.randn() * 1000  # начальное значение смещения
print(f'Начальные веса: w={w:.0f}, b={b:.0f}')

# Гиперпараметры обучения
learning_rate = 1e-10  # попробуйте изменить: 1e-12, 1e-9, 1e-8
n_epochs = 100         # попробуйте изменить: 50, 200, 500

# Нормализуем входы для стабильности обучения
area_mean, area_std = area.mean(), area.std()
area_norm = (area - area_mean) / area_std

price_mean, price_std = price.mean(), price.std()
price_norm = (price - price_mean) / price_std

# Переинициализируем веса для нормализованных данных
w = np.random.randn()
b = np.random.randn()

losses = []

for epoch in range(n_epochs):
    # Прямой проход: прогнозы модели
    predictions = w * area_norm + b
    
    # Функция потерь: среднеквадратичная ошибка (MSE)
    errors = predictions - price_norm
    loss = np.mean(errors ** 2)
    losses.append(loss)
    
    # Градиенты (производные loss по w и b)
    dw = 2 * np.mean(errors * area_norm)
    db = 2 * np.mean(errors)
    
    # Обновление весов: шаг в направлении, уменьшающем loss
    w -= learning_rate * dw
    b -= learning_rate * db
    
    if (epoch + 1) % 20 == 0:
        print(f'Эпоха {epoch+1:3d} | Loss: {loss:.4f} | w={w:.4f}, b={b:.4f}')

print(f'\nОбученные веса: w={w:.4f}, b={b:.4f}')
print(f'(В нормализованном пространстве истинные w~{true_w * area_std / price_std:.4f})')

In [ ]:
# Визуализация результата обучения

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss по эпохам
axes[0].plot(losses)
axes[0].set_title('Функция потерь в процессе обучения')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss (MSE)')
axes[0].grid(True, alpha=0.3)

# Данные + линия регрессии
# Переводим обратно в исходный масштаб
w_orig = w * price_std / area_std
b_orig = b * price_std - w_orig * area_mean + price_mean

x_line = np.linspace(15, 125, 100)
y_line = w_orig * x_line + b_orig

axes[1].scatter(area, price / 1e6, alpha=0.5, s=30, label='Данные')
axes[1].plot(x_line, y_line / 1e6, 'r-', lw=2, label=f'Модель: цена={w_orig/1e3:.0f}k*S + {b_orig/1e6:.1f}M')
axes[1].set_xlabel('Площадь (м2)')
axes[1].set_ylabel('Цена (млн руб.)')
axes[1].set_title('Линейная регрессия: результат обучения')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Модель нашла: цена = {w_orig/1e3:.0f} тыс.руб/м2 * площадь + {b_orig/1e6:.1f} млн руб.')
print(f'Истинная зависимость: цена = {true_w/1e3:.0f} тыс.руб/м2 * площадь + {true_b/1e6:.1f} млн руб.')
print('\nМодель сама «открыла» правила — мы не говорили ей, что зависимость линейная!')

### Что здесь произошло?

1. Мы дали модели **входы** (площадь) и **правильные ответы** (цены).
2. Модель начала со случайных весов `w` и `b` — её прогнозы были ужасными.
3. На каждом шаге она:
   - делала прогноз (`w * площадь + b`),
   - сравнивала его с реальностью (считала loss),
   - вычисляла градиент — «в какую сторону менять веса, чтобы ошибка уменьшилась»,
   - обновляла веса.
4. После 100 эпох модель нашла веса, очень близкие к истинным.

Мы не писали правило «цена = 150 000 * площадь + 500 000». Модель нашла его сама, глядя на данные. Это и есть суть машинного обучения.

**Что можно менять:**
- `learning_rate` — попробуйте 1e-12 (слишком медленно), 1e-8 (может «взорваться»), 1e-10 (хорошо).
- `n_epochs` — больше эпох = точнее, но дольше.
- Уровень шума в данных: измените `np.random.normal(0, 500_000, 100)` на `1_000_000` — модель будет труднее найти закономерность.

---
## Шаг 4. Базовый пример: классификатор «с нуля»

Теперь давайте построим классификатор — модель, которая решает, к какому классу относится объект. Мы вернёмся к нашему примеру с кошками и собаками, но теперь используем **все три признака одновременно** и построим логистическую регрессию — простейший классификатор.

Логистическая регрессия — это линейная модель + сигмоида (функция, которая сжимает выход в диапазон [0, 1]). Выход можно интерпретировать как вероятность класса.

In [ ]:
# Собираем данные: кошки (класс 0) и собаки (класс 1)

np.random.seed(42)
n_per_class = 100

# Кошки: поменьше, покороче шерсть
cats = np.column_stack([
    np.random.uniform(0.1, 0.6, n_per_class),   # размер ушей
    np.random.uniform(0.0, 0.5, n_per_class),   # длина шерсти
    np.random.uniform(0.0, 0.4, n_per_class),   # вес
])

# Собаки: побольше, шерсть длиннее
dogs = np.column_stack([
    np.random.uniform(0.3, 0.9, n_per_class),
    np.random.uniform(0.2, 1.0, n_per_class),
    np.random.uniform(0.4, 1.0, n_per_class),
])

X = np.vstack([cats, dogs])
y = np.array([0] * n_per_class + [1] * n_per_class)  # 0=кошка, 1=собака

print(f'X: {X.shape}, y: {y.shape}')
print(f'Кошек: {(y==0).sum()}, Собак: {(y==1).sum()}')

# Визуализация в 2D (первые два признака)
plt.scatter(X[y==0, 0], X[y==0, 1], alpha=0.6, label='Кошки', s=40)
plt.scatter(X[y==1, 0], X[y==1, 1], alpha=0.6, label='Собаки', s=40)
plt.xlabel('Размер ушей')
plt.ylabel('Длина шерсти')
plt.title('Данные: кошки vs собаки (2 из 3 признаков)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Строим логистическую регрессию вручную!

def sigmoid(z):
    """Сигмоида: сжимает любое число в диапазон (0, 1)."""
    return 1 / (1 + np.exp(-z))

# Нормализация признаков
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_norm = (X - X_mean) / X_std

# Инициализация весов
np.random.seed(42)
W = np.random.randn(3) * 0.1  # 3 веса (по одному на признак)
b = 0.0                        # смещение

# Гиперпараметры
learning_rate = 0.1   # попробуйте изменить: 0.01, 0.5, 1.0
n_epochs = 200        # попробуйте изменить: 50, 100, 500

losses = []
accuracies = []

for epoch in range(n_epochs):
    # Прямой проход
    logits = X_norm @ W + b               # линейная часть
    probs = sigmoid(logits)               # вероятности
    
    # Функция потерь: логарифмическая (binary cross-entropy)
    eps = 1e-7  # чтобы избежать log(0)
    loss = -np.mean(y * np.log(probs + eps) + (1 - y) * np.log(1 - probs + eps))
    losses.append(loss)
    
    # Точность
    preds = (probs >= 0.5).astype(int)
    acc = (preds == y).mean()
    accuracies.append(acc)
    
    # Градиенты
    errors = probs - y
    dW = X_norm.T @ errors / len(y)
    db = errors.mean()
    
    # Обновление весов
    W -= learning_rate * dW
    b -= learning_rate * db
    
    if (epoch + 1) % 50 == 0:
        print(f'Эпоха {epoch+1:3d} | Loss: {loss:.4f} | Accuracy: {acc:.4f}')

print(f'\nИтого: Loss={losses[-1]:.4f}, Accuracy={accuracies[-1]:.4f}')
print(f'Обученные веса: W={W}, b={b:.4f}')

In [ ]:
# Визуализация обучения и границ решений

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss и Accuracy
axes[0].plot(losses, label='Loss', color='red')
axes[0].set_title('Функция потерь')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes2 = axes[0].twinx()
axes2.plot(accuracies, label='Accuracy', color='green', alpha=0.7)
axes2.set_ylabel('Accuracy', color='green')
axes2.tick_params(axis='y', labelcolor='green')

# Граница решений (в пространстве первых двух признаков)
x1_range = np.linspace(X_norm[:, 0].min() - 0.5, X_norm[:, 0].max() + 0.5, 200)
x2_range = np.linspace(X_norm[:, 1].min() - 0.5, X_norm[:, 1].max() + 0.5, 200)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
# Для простоты используем только первые два признака в визуализации
# (третий признак фиксируем на среднем значении)
grid = np.column_stack([xx1.ravel(), xx2.ravel(), np.zeros(xx1.size)])
probs_grid = sigmoid(grid @ W + b).reshape(xx1.shape)

axes[1].contourf(xx1, xx2, probs_grid, alpha=0.3, cmap='RdBu', levels=20)
axes[1].scatter(X_norm[y==0, 0], X_norm[y==0, 1], alpha=0.6, label='Кошки', s=30, edgecolors='k', linewidth=0.5)
axes[1].scatter(X_norm[y==1, 0], X_norm[y==1, 1], alpha=0.6, label='Собаки', s=30, edgecolors='k', linewidth=0.5)
axes[1].set_xlabel('Размер ушей (норм.)')
axes[1].set_ylabel('Длина шерсти (норм.)')
axes[1].set_title('Граница решений классификатора')
axes[1].legend()

plt.tight_layout()
plt.show()

### Разбор: что делает каждый элемент?

- **Логиты** (`X_norm @ W + b`) — линейная комбинация признаков. Это «сырая» оценка принадлежности к классу.
- **Сигмоида** — превращает логит в вероятность от 0 до 1. Если логит = 0, вероятность = 50%. Если логит большой — вероятность близка к 1. Если отрицательный — к 0.
- **Binary cross-entropy** (функция потерь) — штрафует модель за неправильные вероятности. Если правильный класс 1, а модель предсказала 0.1 — большой штраф. Если предсказала 0.9 — маленький.
- **Градиенты** — показывают, как нужно изменить каждый вес, чтобы loss уменьшился.

**Аналогия:** представьте, что вы настраиваете ручки на радио. Каждая ручка — это вес. Градиент говорит: «Покрути эту ручку чуть вправо, и сигнал станет чётче». Вы крутите — и так раз за разом, пока не поймаете станцию.

---
## Шаг 5. Как обучение связано с историей ML

Машинное обучение не появилось вчера. Вот краткая хронология ключевых идей, которые привели к современному deep learning:

| Год | Событие | Суть |
|-----|---------|------|
| 1957 | Перцептрон Розенблатта | Первая искусственная нейросеть: линейный классификатор с обучением |
| 1959 | Артур Сэмюэл | Термин «machine learning» — обучение без явного программирования |
| 1986 | Обратное распространение (Rumelhart et al.) | Алгоритм обучения многослойных сетей через градиенты |
| 1995 | SVM (Вапник) | Метод опорных векторов — мощный классификатор без нейросетей |
| 2012 | AlexNet | Прорыв deep learning в ImageNet — свёрточная сеть обошла все классические методы |
| 2017 | Transformer (Vaswani et al.) | Архитектура «Attention is All You Need» — основа современных LLM |

Ключевое наблюдение: **нейросети существуют с 1950-х**, но стали мощными только недавно благодаря трём факторам:
1. **Данные** — интернет дал терабайты размеченных данных.
2. **Вычисления** — GPU и TPU ускорили обучение в тысячи раз.
3. **Алгоритмы** — улучшенные методы инициализации, регуляризации и оптимизации.

Deep learning — это **подмножество** машинного обучения, которое использует нейросети с многими слоями (отсюда «deep» — глубокие). Но базовые принципы (входы, веса, loss, градиенты, обучение) — одни и те же для всего ML.

In [ ]:
# Визуализация: где находится deep learning в мире ML

fig, ax = plt.subplots(figsize=(10, 7))

# ИИ — самый большой круг
circle_ai = plt.Circle((0.5, 0.5), 0.45, fc='lightyellow', ec='goldenrod', lw=3)
ax.add_patch(circle_ai)
ax.text(0.5, 0.88, 'Искусственный интеллект\n(AI)', ha='center', va='center', fontsize=14, fontweight='bold')

# ML — средний круг
circle_ml = plt.Circle((0.45, 0.42), 0.30, fc='lightblue', ec='steelblue', lw=3)
ax.add_patch(circle_ml)
ax.text(0.45, 0.55, 'Машинное\nобучение\n(ML)', ha='center', va='center', fontsize=13, fontweight='bold', color='steelblue')

# Deep Learning — внутренний круг
circle_dl = plt.Circle((0.40, 0.30), 0.15, fc='lightsalmon', ec='firebrick', lw=3)
ax.add_patch(circle_dl)
ax.text(0.40, 0.30, 'Deep\nLearning', ha='center', va='center', fontsize=12, fontweight='bold', color='firebrick')

# Аннотации
ax.annotate('SVM, деревья,\nk-NN, линейная\nрегрессия...', xy=(0.68, 0.35), fontsize=9,
            ha='center', color='steelblue', fontstyle='italic')
ax.annotate('CNN, RNN,\nTransformers,\nLLM...', xy=(0.40, 0.12), fontsize=9,
            ha='center', color='firebrick', fontstyle='italic')
ax.annotate('Экспертные системы,\nпоиск, планирование...', xy=(0.78, 0.68), fontsize=9,
            ha='center', color='goldenrod', fontstyle='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Иерархия: AI > ML > Deep Learning', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### Перцептрон: прабабушка современных нейросетей

Перцептрон Розенблатта (1957) — это, по сути, та самая логистическая регрессия, которую мы только что построили! Разница лишь в том, что Розенблатт использовал пороговую функцию (step function) вместо сигмоиды. Формула та же: `output = f(w1*x1 + w2*x2 + ... + b)`, где `f` — функция активации.

Проблема перцептрона: он может разделить только **линейно разделимые** данные. Спиральный датасет из предыдущего ноутбука ему не по зубам. Для этого нужны **многослойные** сети — deep learning.

In [ ]:
# Демонстрация: что может и чего не может линейный классификатор

from sklearn.datasets import make_classification, make_moons

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Линейно разделимые данные
X_lin, y_lin = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                   n_informative=2, random_state=42, n_clusters_per_class=1)
axes[0].scatter(X_lin[y_lin==0, 0], X_lin[y_lin==0, 1], alpha=0.6, label='Класс 0')
axes[0].scatter(X_lin[y_lin==1, 0], X_lin[y_lin==1, 1], alpha=0.6, label='Класс 1')
axes[0].set_title('Линейно разделимые\n(перцептрон справится!)', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Нелинейные данные (полумесяцы)
X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)
axes[1].scatter(X_moon[y_moon==0, 0], X_moon[y_moon==0, 1], alpha=0.6, label='Класс 0')
axes[1].scatter(X_moon[y_moon==1, 0], X_moon[y_moon==1, 1], alpha=0.6, label='Класс 1')
axes[1].set_title('Нелинейные данные\n(перцептрон НЕ справится)', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Справа: никакая прямая линия не разделит классы идеально.')
print('Нужны нелинейные модели — нейросети со скрытыми слоями!')

---
## Шаг 6. От ML к глубокому обучению

Deep learning — это ML с многослойными нейросетями. Каждый слой извлекает всё более абстрактные признаки из данных. Например, в задаче распознавания кошек:

- **Слой 1** находит грани (горизонтальные, вертикальные, диагональные линии).
- **Слой 2** комбинирует грани в текстуры и формы (шерсть, глаза, уши).
- **Слой 3** комбинирует формы в части объектов (мордочка, лапы, хвост).
- **Выходной слой** собирает всё вместе и говорит: «Это кошка с вероятностью 92%».

Нам не нужно программировать эти этапы вручную — сеть учится сама. Мы просто даём данные и достаточное количество слоёв.

Давайте посмотрим, как добавление скрытого слоя позволяет решить нелинейную задачу:

In [ ]:
# Сравниваем: линейная модель vs нейросеть на нелинейных данных

from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Данные: полумесяцы (нелинейная задача)
X_moon, y_moon = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_moon, y_moon, test_size=0.2, random_state=42)

# Логистическая регрессия (линейная модель)
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)
lr_acc = lr_model.score(X_test, y_test)
print(f'Логистическая регрессия: accuracy = {lr_acc:.4f}')

# Нейросеть с одним скрытым слоем
# попробуйте изменить hidden_layer_sizes: (16,), (64, 32), (128, 64, 32)
nn_model = MLPClassifier(hidden_layer_sizes=(32,), activation='relu',
                         max_iter=1000, random_state=42)
nn_model.fit(X_train, y_train)
nn_acc = nn_model.score(X_test, y_test)
print(f'Нейросеть (1 скрытый слой): accuracy = {nn_acc:.4f}')

# Нейросеть с двумя скрытыми слоями
nn_model2 = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                          max_iter=1000, random_state=42)
nn_model2.fit(X_train, y_train)
nn_acc2 = nn_model2.score(X_test, y_test)
print(f'Нейросеть (2 скрытых слоя): accuracy = {nn_acc2:.4f}')

In [ ]:
# Визуализация границ решений

def plot_decision_sklearn(model, X, y, ax, title):
    """Рисуем границу решений для sklearn-модели."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='Set2', edgecolors='k', s=20, linewidth=0.5)
    ax.set_title(title, fontsize=12)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_decision_sklearn(lr_model, X_moon, y_moon, axes[0],
                      f'Логистическая регрессия\nAccuracy: {lr_acc:.3f}')
plot_decision_sklearn(nn_model, X_moon, y_moon, axes[1],
                      f'Нейросеть: 1 скрытый слой\nAccuracy: {nn_acc:.3f}')
plot_decision_sklearn(nn_model2, X_moon, y_moon, axes[2],
                      f'Нейросеть: 2 скрытых слоя\nAccuracy: {nn_acc2:.3f}')

plt.tight_layout()
plt.show()

print('Разница видна невооружённым глазом!')
print('Линейная модель проводит ПРЯМУЮ границу — и ошибается на изгибах.')
print('Нейросеть с 1 слоем уже строит ГНУТУЮ границу.')
print('Нейросеть с 2 слоями ещё точнее повторяет форму данных.')

### Почему слои — это «глубина»?

Каждый скрытый слой — это нелинейное преобразование. Комбинируя слои, мы строим всё более сложные функции. Математически это называется **универсальной аппроксимацией**: нейросеть с достаточным числом нейронов в одном скрытом слое может аппроксимировать любую непрерывную функцию. На практике для сложных функций удобнее использовать несколько слоёв поменьше, чем один гигантский.

Аналогия: один мастер-плотник может построить дом, но бригада из нескольких специалистов (фундаментщик, кровельщик, отделочник) сделает это быстрее и лучше. Так же и со слоями: каждый специализируется на своём уровне абстракции.

**Что можно менять:**
- `hidden_layer_sizes` — попробуйте `(8,)`, `(128,)`, `(64, 32, 16)`. Как меняется граница решений?
- `activation` — замените `'relu'` на `'tanh'` или `'logistic'` (сигмоида).
- `noise` в `make_moons` — увеличьте до 0.3 или 0.4. Станет ли нейросети труднее?

---
## Шаг 7. Практические задания / Эксперименты

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

### Задание 1. Когда ML не нужен

Ниже перечислены задачи. Определите, какие из них требуют ML, а какие решаются обычным программированием. Объясните почему.

1. Подсчёт суммы элементов в списке
2. Распознавание речи (голос -> текст)
3. Проверка email-адреса на корректность (regex)
4. Рекомендация фильмов пользователю
5. Сортировка массива
6. Определение языка текста
7. Вычисление факториала
8. Автопилот автомобиля

**Подсказка:** спросите себя — «Могу ли я написать точные правила для этой задачи?» Если да — обычный код. Если правила слишком сложны или их невозможно сформулировать — нужен ML.

### Задание 2. Исследуйте learning rate

Вернитесь к коду обучения логистической регрессии (шаг 4) и попробуйте три значения `learning_rate`: **0.01**, **0.1**, **1.0**.

Для каждого значения:
- Запишите финальный loss и accuracy.
- Постройте графики loss по эпохам на одном рисунке.

**Подсказка:** сохраняйте `losses` в словарь `results = {0.01: [], 0.1: [], 1.0: []}`. Что происходит при слишком большом lr?

### Задание 3. Добавьте признак

В нашем классификаторе кошек/собак есть 3 признака. Добавьте 4-й: «игривость» (0-1). Кошки более игривы (0.4-1.0), собаки менее (0.0-0.6). Переобучите модель и сравните accuracy.

**Подсказка:** добавьте столбец в `cats` и `dogs`, обновите размерность `W = np.random.randn(4)`. Улучшился ли результат? Всегда ли больше признаков = лучше?

### Задание 4. Найдите ошибку в обучении

Вот сломанный код. Найдите и исправьте **3 ошибки**:

```python
W = np.random.randn(3)
b = 0
lr = 0.1

for epoch in range(100):
    logits = X_norm @ W + b
    probs = sigmoid(logits)
    loss = -np.mean(y * np.log(probs) + (1 - y) * np.log(1 - probs))
    # Ошибка 1: что забыли перед backward-шагом?
    errors = probs - y
    W = W - lr * (X_norm.T @ errors / len(y))  # Ошибка 2: как обновляются веса?
    b = b - lr * errors.mean()
    # Ошибка 3: что будет с loss, если probs содержит 0 или 1?
```

**Подсказка:** вспомните про численную стабильность log(0) и про то, как работает присваивание в NumPy (in-place vs создание нового массива).

### Задание 5. Визуализируйте процесс обучения

Модифицируйте код обучения нейросети из шага 6 так, чтобы каждые 50 итераций сохранять текущую границу решений. Создайте сетку из 4-6 графиков, показывающую, как граница решений эволюционирует в процессе обучения.

**Подсказка:**
```python
from sklearn.neural_network import MLPClassifier
# Используйте warm_start=True и partial_fit()
# Или обучайте модель в цикле, увеличивая max_iter
```

---
## Шаг 8. Идеи для дальнейшего изучения

Вы разобрались с фундаментом! Вот куда двигаться дальше:

- **Библиотеки ML**: scikit-learn (классический ML), PyTorch / TensorFlow (deep learning), FastAI (быстрый старт с DL).
- **Типы задач ML**: регрессия, классификация, кластеризация, генерация, обучение с подкреплением.
- **Оценка моделей**: train/test split, кросс-валидация, метрики (precision, recall, F1, ROC-AUC).
- **Переобучение и регуляризация**: почему модель может слишком хорошо выучить тренировочные данные и плохо работать на новых.
- **Работа с данными**: очистка, нормализация, feature engineering — часто важнее, чем выбор модели.

**Рекомендуемый путь:**
1. Ноутбук «Введение в PyTorch» (из этого репозитория)
2. FastAI — курс «Practical Deep Learning for Coders» (бесплатный, очень практичный)
3. Книга «Hands-On Machine Learning» (Aurelien Geron) — от классического ML к deep learning

**Ключевые ресурсы:**
- [fast.ai](https://www.fast.ai/) — практические курсы по deep learning
- [Google ML Crash Course](https://developers.google.com/machine-learning/crash-course) — короткий курс от Google
- [3Blue1Brown: Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) — лучшая визуализация того, как работают нейросети

Удачи на пути в машинное обучение!